In [77]:
# =========================
# Imports Projeto
# =========================

import sys
from pathlib import Path

# ------------------------------------------
# Sobe um nível a partir de jobs/
# para encontrar a pasta src
# ------------------------------------------

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.config.secrets import get_secret

# =========================
# Container Bronze
# =========================

BRONZE_CONTAINER = get_secret(
    "AZURE-CONTAINER-BRONZE",
    "AZURE_CONTAINER_BRONZE"
)

# =========================
# Validação
# =========================

if not BRONZE_CONTAINER:
    raise ValueError(
        "❌ Container Bronze não configurado."
    )

# =========================
# Log
# =========================

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

✅ Container Bronze: bronze


In [78]:
# =========================
# Imports Projeto
# =========================

import sys
from pathlib import Path

# sobe um nível a partir de jobs/

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.config.secrets import get_secret

# =========================
# Imports
# =========================

from azure.storage.blob import BlobServiceClient

# =========================
# Azure Blob
# =========================

try:

    # ------------------------------------------
    # Azure Storage
    # ------------------------------------------

    AZURE_STORAGE_ACCOUNT = get_secret(
        "AZURE-STORAGE-ACCOUNT",
        "AZURE_STORAGE_ACCOUNT"
    )

    AZURE_STORAGE_KEY = get_secret(
        "AZURE-STORAGE-KEY",
        "AZURE_STORAGE_KEY"
    )

    # ------------------------------------------
    # Validações
    # ------------------------------------------

    if not AZURE_STORAGE_ACCOUNT:
        raise ValueError(
            "❌ AZURE-STORAGE-ACCOUNT não configurada."
        )

    if not AZURE_STORAGE_KEY:
        raise ValueError(
            "❌ AZURE-STORAGE-KEY não configurada."
        )

    # ------------------------------------------
    # URL Storage Account
    # ------------------------------------------

    account_url = (
        f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net"
    )

    # ------------------------------------------
    # Cliente Blob Storage
    # ------------------------------------------

    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=AZURE_STORAGE_KEY
    )

    # ------------------------------------------
    # Logs
    # ------------------------------------------

    print(
        "✅ Cliente Azure Blob inicializado."
    )

    print(
        f"✅ Storage Account: "
        f"{AZURE_STORAGE_ACCOUNT}"
    )

    print(
        f"✅ Account URL: "
        f"{account_url}"
    )

except Exception as e:

    print(
        f"❌ Erro Azure Blob: {e}"
    )

    raise

✅ Cliente Azure Blob inicializado.
✅ Storage Account: stfiapoin4ci2kb4w7c
✅ Account URL: https://stfiapoin4ci2kb4w7c.blob.core.windows.net


In [79]:
# =========================
# API IBGE
# =========================

import requests

# ------------------------------------------
# Endpoint IBGE
# ------------------------------------------

url = (
    "https://servicodados.ibge.gov.br/"
    "api/v1/localidades/estados"
)

# ------------------------------------------
# Requisição
# ------------------------------------------

response = requests.get(
    url,
    timeout=60
)

response.raise_for_status()

# ------------------------------------------
# JSON
# ------------------------------------------

data = response.json()

# ------------------------------------------
# Validação
# ------------------------------------------

if not data:
    raise ValueError(
        "❌ Nenhum estado retornado pela API do IBGE."
    )

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Estados encontrados: "
    f"{len(data)}"
)


✅ Estados encontrados: 27


In [80]:
# =========================
# DataFrame
# =========================

import pandas as pd
import datetime as dt

# ------------------------------------------
# Validação API
# ------------------------------------------

if not isinstance(data, list):
    raise ValueError(
        f"❌ Tipo inesperado retornado pela API: {type(data)}"
    )

if len(data) == 0:
    raise ValueError(
        "❌ API retornou lista vazia."
    )

# ------------------------------------------
# DataFrame
# ------------------------------------------

df = pd.json_normalize(data)

# ------------------------------------------
# Renomeia colunas
# ------------------------------------------

df = df.rename(
    columns={
        "id": "estado_id",
        "sigla": "estado_sigla",
        "nome": "estado_nome",
        "regiao.id": "regiao_id",
        "regiao.sigla": "regiao_sigla",
        "regiao.nome": "regiao_nome"
    }
)

# ------------------------------------------
# Metadados de ingestão
# ------------------------------------------

df["_ingested_at"] = (
    dt.datetime.now(
        dt.timezone.utc
    ).isoformat()
)

# ------------------------------------------
# Ordenação
# ------------------------------------------

df = (
    df
    .sort_values(
        by="estado_sigla"
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------
# Validação DataFrame
# ------------------------------------------

if df.empty:
    raise ValueError(
        "❌ DataFrame IBGE Estados vazio."
    )

required_columns = [
    "estado_id",
    "estado_sigla",
    "estado_nome",
    "regiao_id",
    "regiao_sigla",
    "regiao_nome",
    "_ingested_at"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"❌ Colunas ausentes: {missing_columns}"
    )

# ------------------------------------------
# Tipos de dados
# ------------------------------------------

df["estado_id"] = df["estado_id"].astype(int)
df["regiao_id"] = df["regiao_id"].astype(int)

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Registros carregados: "
    f"{len(df)}"
)

print(
    f"✅ Colunas carregadas: "
    f"{len(df.columns)}"
)

print(
    f"✅ Estados únicos: "
    f"{df['estado_sigla'].nunique()}"
)

print(
    f"✅ Regiões únicas: "
    f"{df['regiao_sigla'].nunique()}"
)

print(
    f"✅ Timestamp de ingestão: "
    f"{df['_ingested_at'].iloc[0]}"
)

# ------------------------------------------
# Preview
# ------------------------------------------

display(df)

✅ Registros carregados: 27
✅ Colunas carregadas: 7
✅ Estados únicos: 27
✅ Regiões únicas: 5
✅ Timestamp de ingestão: 2026-08-26T08:33:07.969284+00:00


,estado_id,estado_sigla,estado_nome,regiao_id,regiao_sigla,regiao_nome,_ingested_at
0,12,AC,Acre,1,N,Norte,2026-08-26T08:33:07.969284+00:00
1,27,AL,Alagoas,2,NE,Nordeste,2026-08-26T08:33:07.969284+00:00
2,13,AM,Amazonas,1,N,Norte,2026-08-26T08:33:07.969284+00:00
3,16,AP,Amapá,1,N,Norte,2026-08-26T08:33:07.969284+00:00
4,29,BA,Bahia,2,NE,Nordeste,2026-08-26T08:33:07.969284+00:00
5,23,CE,Ceará,2,NE,Nordeste,2026-08-26T08:33:07.969284+00:00
6,53,DF,Distrito Federal,5,CO,Centro-Oeste,2026-08-26T08:33:07.969284+00:00
7,32,ES,Espírito Santo,3,SE,Sudeste,2026-08-26T08:33:07.969284+00:00
8,52,GO,Goiás,5,CO,Centro-Oeste,2026-08-26T08:33:07.969284+00:00
9,21,MA,Maranhão,2,NE,Nordeste,2026-08-26T08:33:07.969284+00:00


In [81]:
# =========================
# Salvar Parquet
# =========================

from pathlib import Path
import datetime as dt

# ------------------------------------------
# Validação DataFrame
# ------------------------------------------

if df.empty:
    raise ValueError(
        "❌ DataFrame vazio. Nada para salvar."
    )

# ------------------------------------------
# Diretório temporário
# ------------------------------------------

temp_dir = Path.cwd() / "tmp"

temp_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------
# Nome do arquivo
# ------------------------------------------

date_suffix = dt.datetime.now().strftime(
    "%Y-%m-%d"
)

parquet_file = (
    temp_dir
    / f"{date_suffix}_ibge_estados.parquet"
)

# ------------------------------------------
# Salva Parquet
# ------------------------------------------

df.to_parquet(
    parquet_file,
    index=False
)

# ------------------------------------------
# Valida criação
# ------------------------------------------

if not parquet_file.exists():
    raise FileNotFoundError(
        f"❌ Arquivo não foi criado: {parquet_file}"
    )

file_size_bytes = parquet_file.stat().st_size
file_size_kb = round(
    file_size_bytes / 1024,
    2
)

if file_size_bytes == 0:
    raise ValueError(
        f"❌ Arquivo criado sem conteúdo: {parquet_file}"
    )

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Parquet criado: "
    f"{parquet_file}"
)

print(
    f"✅ Nome arquivo: "
    f"{parquet_file.name}"
)

print(
    f"✅ Total de registros: "
    f"{len(df)}"
)

print(
    f"✅ Total de colunas: "
    f"{len(df.columns)}"
)

print(
    f"✅ Tamanho do arquivo: "
    f"{file_size_kb} KB"
)

print(
    f"✅ Diretório temporário: "
    f"{temp_dir}"
)

✅ Parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp/2026-08-26_ibge_estados.parquet
✅ Nome arquivo: 2026-08-26_ibge_estados.parquet
✅ Total de registros: 27
✅ Total de colunas: 7
✅ Tamanho do arquivo: 5.2 KB
✅ Diretório temporário: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp


In [82]:
# =========================
# Upload Bronze
# =========================

# ------------------------------------------
# Validações
# ------------------------------------------

if df.empty:
    raise ValueError(
        "❌ DataFrame vazio. Nada para enviar."
    )

if not parquet_file.exists():
    raise FileNotFoundError(
        f"❌ Arquivo não encontrado: {parquet_file}"
    )

if not BRONZE_CONTAINER:
    raise ValueError(
        "❌ Container Bronze não configurado."
    )

if not blob_service_client:
    raise ValueError(
        "❌ Cliente Azure Blob não inicializado."
    )

# ------------------------------------------
# Nome do blob
# ------------------------------------------

blob_name = (
    f"{date_suffix}_ibge_estados.parquet"
)

# ------------------------------------------
# Cliente Blob
# ------------------------------------------

blob_client = (
    blob_service_client.get_blob_client(
        container=BRONZE_CONTAINER,
        blob=blob_name
    )
)

# ------------------------------------------
# Upload
# ------------------------------------------

with open(
    parquet_file,
    "rb"
) as file_data:

    blob_client.upload_blob(
        file_data,
        overwrite=True
    )

# ------------------------------------------
# Validação Upload
# ------------------------------------------

if not blob_client.exists():
    raise RuntimeError(
        f"❌ Upload não encontrado no container: {blob_name}"
    )

blob_properties = (
    blob_client.get_blob_properties()
)

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Upload concluído: "
    f"{blob_name}"
)

print(
    f"✅ Container: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Arquivo local: "
    f"{parquet_file}"
)

print(
    f"✅ Registros enviados: "
    f"{len(df)}"
)

print(
    f"✅ Tamanho enviado: "
    f"{round(blob_properties.size / 1024, 2)} KB"
)

print(
    f"✅ Blob validado no Azure Storage"
)

✅ Upload concluído: 2026-08-26_ibge_estados.parquet
✅ Container: bronze
✅ Arquivo local: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp/2026-08-26_ibge_estados.parquet
✅ Registros enviados: 27
✅ Tamanho enviado: 5.2 KB
✅ Blob validado no Azure Storage
